In [108]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [188]:
def efficient_frontier(means, cov, pos_weights=True, Rp=None, vol=None, custom_min_func = None):
    """
        Custom mean function must be a 3 input function, func(x, mean, cov), as this function
        doesn't innately know what needs to be passed to it, so everything is passed
    """
    if Rp is None and vol is None and custom_min_func is None:
        raise ValueError("One of returns or volatility must be given")
    
    if Rp is not None and vol is not None:
        raise ValueError("Only one of returns or volatility can be given")
    
    def var(w, cov):
        return w.T @ cov @ w
    
    def ret(w, means):
        return -np.sum(w*means)

    w_constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    constraints = [w_constraint]

    if pos_weights:
        bounds = ((0, None),)*cov.shape[0]
    else:
        bounds = None
    
    if Rp is not None:
        ret_constraint = {'type': 'eq', 'fun': lambda w: np.sum(w*means) - Rp}
        constraints.append(ret_constraint)
        min_function = lambda x: var(x,cov)
    elif vol is not None:
        vol_constraint = {'type': 'eq', 'fun': lambda w: (w.T @ cov @ w) - vol**2}
        constraints.append(vol_constraint)
        min_function = lambda x: ret(x, means)
    
    if custom_min_func is not None:
        min_function = lambda x: custom_min_func(x, means, cov)
    
    result = minimize(fun=min_function, x0=np.ones(means.shape[0])/means.shape[0], method='SLSQP', bounds=bounds, constraints=constraints, options={'ftol': 1e-20, 'maxiter': 1000, 'disp': False})
    
    return result.x

In [231]:
data_5_1 = pd.read_csv("testfiles/data/test5_1.csv")
cov = data_5_1.to_numpy()
means = np.array([0.21, 0.07, -0.01, 0.07, 0.05])

In [249]:
means, cov

(array([ 0.21,  0.07, -0.01,  0.07,  0.05]),
 array([[0.08497905, 0.08758581, 0.04230441, 0.00898354, 0.00387595],
        [0.08758581, 0.16048451, 0.05813615, 0.01234549, 0.00532646],
        [0.04230441, 0.05813615, 0.03744009, 0.00596294, 0.00257271],
        [0.00898354, 0.01234549, 0.00596294, 0.00168834, 0.00054633],
        [0.00387595, 0.00532646, 0.00257271, 0.00054633, 0.00031428]]))

In [246]:
def sharpe_ratio(w, means, cov, r):
    return (w @ means - r) / np.sqrt(w.T @ cov @ w)

rfr = 0.04
capital_market_portfolio = efficient_frontier(means, cov, pos_weights=True, custom_min_func= lambda a, b, c: -sharpe_ratio(a, b, c, rfr))
print(capital_market_portfolio)
print(sharpe_ratio(capital_market_portfolio, means, cov, rfr))

[0.0163816  0.         0.         0.97187352 0.01174488]
0.7320961700096917


### Using the efficient frontier framework but optimizing on other things

In [253]:
from risk_management import risk_metrics

def expected_shortfall(w, means, cov, alpha = 0.05):
    mu = w @ means
    sigma = np.sqrt(w.T @ cov @ w)
    abs_ES, _ = risk_metrics.expected_shortfall_normal(mu, sigma, alpha)
    return abs_ES

optimal_es_portfolio = efficient_frontier(means, cov, pos_weights=True, custom_min_func= lambda a, b, c: expected_shortfall(a,b,c, alpha=0.05))
print(optimal_es_portfolio)
print(expected_shortfall(optimal_es_portfolio, means, cov))


[2.04012784e-17 0.00000000e+00 3.78189205e-17 2.67970941e-17
 1.00000000e+00]
-0.013432130635563103
0.3913048376791972


In [255]:
from risk_management import risk_metrics

def VaR(w, means, cov, alpha = 0.05):
    mu = w @ means
    sigma = np.sqrt(w.T @ cov @ w)
    abs_VaR, _ = risk_metrics.univariate_normal_VaR(mu, sigma, alpha)
    return abs_VaR

optimal_var_portfolio = efficient_frontier(means, cov, pos_weights=True, custom_min_func= lambda a, b, c: VaR(a,b,c, alpha=0.05))
print(optimal_var_portfolio)
print(VaR(optimal_var_portfolio, means, cov))


[3.78081496e-17 0.00000000e+00 0.00000000e+00 1.22186287e-15
 1.00000000e+00]
-0.020839957780324613


### Risk Parity with ES and VAR
#### Euler allocation

#### NOTE -> Can't do risk parity with VAR because it's noncoherent

In [276]:
from risk_management import portfolio_construction
from scipy.stats import norm

def min_sse_ces(w, cov, risk_budgets, alpha=0.05):
    z_alpha = norm.ppf(alpha)
    ev_given_z_lt_z_alpha = (-norm.pdf(z_alpha) / alpha)

    m = cov @ w
    vol = np.sqrt(w.T @ cov @ w)
    
    ES = -(vol * ev_given_z_lt_z_alpha)
    cES = w * m * ev_given_z_lt_z_alpha / vol
    cES_budgeted = cES / risk_budgets
    return np.sum((cES_budgeted - np.mean(cES_budgeted))**2)

risk_budgets = np.array([1,1,1,1,1])
ES_risk_parity_weights = portfolio_construction.compute_risk_parity_weights(cov, risk_budgets, custom_objective_func=lambda x, y, z: min_sse_ces(x, y, z, alpha=0.05))
ES_risk_parity_weights, risk_metrics.expected_shortfall_normal(ES_risk_parity_weights @ means, ES_risk_parity_weights.T @ cov @ ES_risk_parity_weights)[0]

Optimization terminated successfully    (Exit mode 0)
            Current function value: 2.3854137945734264e-17
            Iterations: 18
            Function evaluations: 119
            Gradient evaluations: 18


(array([0.03735199, 0.02718023, 0.05627315, 0.26499599, 0.61419863]),
 np.float64(-0.05355233511240011))

In [3]:
import numpy as np
test=np.array([[0.02, 0.03],
         [0.02, 0.03], 
         [0.02, 0.03]])

In [10]:
np.cumprod(1+test, axis=0)-1

array([[0.02    , 0.03    ],
       [0.0404  , 0.0609  ],
       [0.061208, 0.092727]])

### We already did normal var in risk_metrics, but let's explicitely do delta normal VaR

In [324]:
def delta_normal_var(asset_prices, weights, underlying_prices, asset_underlying_price_indices, deltas, cov, alpha = 0.05):
    portfolio_value = weights @ asset_prices

    # d asset return / d underlying return (often 1 if asset is the underlying)
    dRA_dri = deltas * underlying_prices[asset_underlying_price_indices] / asset_prices

    # how much each underlying contributes to the portfolio returns
    weight_contributions = np.zeros((underlying_prices.shape[0], weights.shape[0]))
    col_indices = np.arange(weights.shape[0])
    weight_contributions[asset_underlying_price_indices, col_indices] = weights

    grad_r = weight_contributions @ dRA_dri # dR / dri -> d portfolio return / d underlying return

    sigma_p = np.sqrt(grad_r.T @ cov @ grad_r)
    VaR = - portfolio_value * norm.ppf(alpha) * sigma_p

    return VaR

In [303]:
# 2 ways to do it

# REMEMBER THAT THESE VALUES ARE D_RETURNS

underlying_prices = np.array([10, 20, 30, 40]) # the underlying assets whose prices may move

asset_prices = np.array([10, 20, 30, 40, 0.8]) 
deltas = np.array([1,1,1,1,0.5]) # last one is a call option on the first asset # TODO -> not called delta because it's about percents
price_indices = np.array([0,1,2,3,0]) # the underlying asset price we have to be worried about
# weights way
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2]) # weights of the assets

portfolio_value = weights @ asset_prices

dRA_dri = deltas * underlying_prices[price_indices] / asset_prices # d asset returns / d underlying returns


weight_contributions = np.zeros((underlying_prices.shape[0], weights.shape[0]))
col_indices = np.arange(weights.shape[0])
weight_contributions[price_indices, col_indices] = weights


# dRa_dr = deltas * underlying_prices
grad_r = weight_contributions @ dRA_dri # dR / dri -> d portfolio return / d asset return
print(grad_r)

VaR = - portfolio_value * norm.ppf(alpha) * np.sqrt(grad_r.T @ cov @ grad_r)

# d_asset_price_d_price = delta* price / asset_price
# dR_dri = sum(weights * deltas*prices[i] / asset_prices)

# dR_dri = prices[i] * sum (weights  * deltas/ asset_returns) 


# holdings way


holdings = np.array([1, 2, 3, 4, 5])

# dR_dri = prices[i] / portfolio_value * sum(holdings * deltas) # d Return / d asset return is sum of all assets attached to that underlying


# VaR = - portfolio_value * norm.ppf(alpha) * np.sqrt(grad_r.T @ cov @ grad_r)

[1.45 0.2  0.2  0.2 ]


In [323]:
asset_prices = np.array([10, 20, 30, 40, 0.8]) 
underlying_prices = np.array([10, 20, 30, 40]) # the underlying assets whose prices may move
price_indices = np.array([0,1,2,3,0]) # the underlying asset price we have to be worried about
deltas = np.array([1,1,1,1,0.5]) # last one is a call option on the first asset # TODO -> not called delta because it's about percents
cov = data_5_1.to_numpy()[:-1, :-1]

# weights way
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2]) # weights of the assets in the portfolio
portfolio_value = weights @ asset_prices

# holdings way
# holdings = np.array([4, 2, 4/3, 1, 50])
# w = holdings * asset_prices / (holdings @ asset_prices)
# print(w)

VaR = delta_normal_var(asset_prices, weights, underlying_prices, price_indices, deltas, cov, alpha=0.05)

In [331]:
def delta_normal_es(asset_prices, weights, underlying_prices, asset_underlying_price_indices, deltas, cov, alpha=0.05):
    portfolio_value = weights @ asset_prices

    # d asset return / d underlying return (often 1 if asset is the underlying)
    dRA_dri = deltas * underlying_prices[asset_underlying_price_indices] / asset_prices

    # how much each underlying contributes to the portfolio returns
    weight_contributions = np.zeros((underlying_prices.shape[0], weights.shape[0]))
    col_indices = np.arange(weights.shape[0])
    weight_contributions[asset_underlying_price_indices, col_indices] = weights

    grad_r = weight_contributions @ dRA_dri # dR / dri -> d portfolio return / d underlying return

    # zero-mean ES closed form
    z = norm.ppf(alpha)
    pdf_z = norm.pdf(z)
    sigma_p = np.sqrt(grad_r.T @ cov @ grad_r)

    ES = portfolio_value * sigma_p * pdf_z / alpha

    # ES = - portfolio_value * norm.ppf(alpha) * np.sqrt(grad_r.T @ cov @ grad_r)

    return ES

In [332]:
ES = delta_normal_es(asset_prices, weights, underlying_prices, price_indices, deltas, cov, alpha=0.05)
ES

[1.   1.   1.   1.   6.25]


np.float64(21.740087153471496)